In [ ]:
import os
import gc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
pd.set_option("display.max_columns", None)

In [ ]:
from plot_utils import topk_barplot, cumsum_plot

In [ ]:
sample_size = 25

In [ ]:
current_dir = os.getcwd()

data_dir = "tech_challenge-data-us_flights_ml"

In [ ]:
flights_csv = "flights.csv"
df_flights = pd.read_csv(os.path.join(current_dir, data_dir, flights_csv))

In [ ]:
df_flights.info()

In [ ]:
not_cancelled_mask = df_flights["CANCELLED"] == 0

df_flights.loc[not_cancelled_mask, "FLIGHT_SEQUENCE"] = (
    df_flights.loc[not_cancelled_mask, ["TAIL_NUMBER", "YEAR", "MONTH", "DAY", "SCHEDULED_DEPARTURE"]]
    .sort_values(by=["TAIL_NUMBER", "YEAR", "MONTH", "DAY", "SCHEDULED_DEPARTURE"])
    .groupby(["TAIL_NUMBER", "YEAR", "MONTH", "DAY"])
    .cumcount() + 1
)

df_flights["FLIGHT_SEQUENCE"] = df_flights["FLIGHT_SEQUENCE"].astype("Int64")

In [ ]:
df_flights.loc[df_flights["TAIL_NUMBER"] == "N3KUAA", ["YEAR", "MONTH", "DAY", "SCHEDULED_DEPARTURE", "AIRLINE", "FLIGHT_NUMBER", "TAIL_NUMBER", "ORIGIN_AIRPORT", "DESTINATION_AIRPORT", "FLIGHT_SEQUENCE"]].head(n=sample_size)

In [ ]:
df_flights_considered = (
    df_flights[(df_flights["CANCELLED"] == 0) & (df_flights["DIVERTED"] == 0)]
    .drop(columns=["CANCELLED", "CANCELLATION_REASON", "DIVERTED"])
)

In [ ]:
del df_flights
gc.collect()

In [ ]:
df_flights_considered.info()

In [ ]:
df_flights_considered.sample(n=sample_size)

In [ ]:
wd_mapping = {
    1: "MON",
    2: "TUE",
    3: "WED",
    4: "THU",
    5: "FRI",
    6: "SAT",
    7: "SUN",
}

df_flights_considered["DAY_OF_WEEK_ABBR"] = pd.Categorical(
    df_flights_considered["DAY_OF_WEEK"],
    categories=wd_mapping.keys(),
    ordered=True
).rename_categories(wd_mapping)

In [ ]:
df_flights_considered["ROUTE"] = df_flights_considered["ORIGIN_AIRPORT"].astype(str) + "-" + df_flights_considered["DESTINATION_AIRPORT"].astype(str)

## Sanity checks

In [ ]:
df_flights_considered[["SCHEDULED_DEPARTURE", "SCHEDULED_ARRIVAL", "SCHEDULED_TIME", "DEPARTURE_TIME", "DEPARTURE_DELAY", "ARRIVAL_TIME", "ELAPSED_TIME", "ARRIVAL_DELAY"]].isna().sum()

In [ ]:
df_flights_considered["HOUR"] = df_flights_considered["SCHEDULED_DEPARTURE"] // 100
df_flights_considered["MINUTE"] = df_flights_considered["SCHEDULED_DEPARTURE"] % 100
df_flights_considered["DT_SCHEDULED_DEPARTURE"] = pd.to_datetime(df_flights_considered[["YEAR", "MONTH", "DAY", "HOUR", "MINUTE"]])

df_flights_considered["TD_SCHEDULED_TIME"] = pd.to_timedelta(df_flights_considered["SCHEDULED_TIME"], unit="min")
df_flights_considered["TD_DEPARTURE_DELAY"] = pd.to_timedelta(df_flights_considered["DEPARTURE_DELAY"], unit="min")
df_flights_considered["TD_ELAPSED_TIME"] = pd.to_timedelta(df_flights_considered["ELAPSED_TIME"], unit="min")

df_flights_considered["DT_SCHEDULED_ARRIVAL_CALC"] = df_flights_considered["DT_SCHEDULED_DEPARTURE"] + df_flights_considered["TD_SCHEDULED_TIME"]
df_flights_considered["DT_ARRIVAL_CALC"] = df_flights_considered["DT_SCHEDULED_DEPARTURE"] + df_flights_considered["TD_DEPARTURE_DELAY"] + df_flights_considered["TD_ELAPSED_TIME"]

df_flights_considered["INT_ARRIVAL_DELAY_CALC_1"] = (df_flights_considered["DT_ARRIVAL_CALC"] - df_flights_considered["DT_SCHEDULED_ARRIVAL_CALC"]).dt.total_seconds() / 60
df_flights_considered["INT_ARRIVAL_DELAY_CALC_2"] = (df_flights_considered["TD_DEPARTURE_DELAY"] + df_flights_considered["TD_ELAPSED_TIME"] - df_flights_considered["TD_SCHEDULED_TIME"]).dt.total_seconds() / 60

In [ ]:
df_flights_considered[["DT_SCHEDULED_DEPARTURE", "TD_SCHEDULED_TIME", "DT_SCHEDULED_ARRIVAL_CALC", "SCHEDULED_ARRIVAL"]].sample(n=sample_size)

In [ ]:
(df_flights_considered["DT_SCHEDULED_ARRIVAL_CALC"].dt.minute != (df_flights_considered["SCHEDULED_ARRIVAL"] % 100)).sum()

In [ ]:
df_flights_considered.loc[
    df_flights_considered["DT_SCHEDULED_ARRIVAL_CALC"].dt.minute != (df_flights_considered["SCHEDULED_ARRIVAL"] % 100),
    ["AIRLINE", "FLIGHT_NUMBER", "TAIL_NUMBER", "ORIGIN_AIRPORT", "DESTINATION_AIRPORT", "DT_SCHEDULED_DEPARTURE", "TD_SCHEDULED_TIME", "DT_SCHEDULED_ARRIVAL_CALC", "SCHEDULED_ARRIVAL"]
]

In [ ]:
(df_flights_considered["DT_SCHEDULED_ARRIVAL_CALC"].dt.hour - (df_flights_considered["SCHEDULED_ARRIVAL"] // 100)).value_counts()

In [ ]:
df_flights_considered[["DT_SCHEDULED_DEPARTURE", "TD_DEPARTURE_DELAY", "DEPARTURE_TIME", "TD_ELAPSED_TIME", "DT_ARRIVAL_CALC", "ARRIVAL_TIME"]].sample(n=sample_size)

In [ ]:
(df_flights_considered["DT_ARRIVAL_CALC"].dt.minute != (df_flights_considered["ARRIVAL_TIME"] % 100)).sum()

In [ ]:
df_flights_considered.loc[
    df_flights_considered["DT_ARRIVAL_CALC"].dt.minute != (df_flights_considered["ARRIVAL_TIME"] % 100),
    ["AIRLINE", "FLIGHT_NUMBER", "TAIL_NUMBER", "ORIGIN_AIRPORT", "DESTINATION_AIRPORT", "DT_SCHEDULED_DEPARTURE", "TD_DEPARTURE_DELAY", "DEPARTURE_TIME", "TD_ELAPSED_TIME", "DT_ARRIVAL_CALC", "ARRIVAL_TIME"]
]

In [ ]:
(df_flights_considered["DT_ARRIVAL_CALC"].dt.hour - (df_flights_considered["ARRIVAL_TIME"] // 100)).value_counts()

In [ ]:
df_flights_considered[["ARRIVAL_TIME", "SCHEDULED_ARRIVAL", "ARRIVAL_DELAY", "INT_ARRIVAL_DELAY_CALC_1", "INT_ARRIVAL_DELAY_CALC_2"]].sample(n=sample_size)

In [ ]:
delay_cols = ["ARRIVAL_DELAY", "INT_ARRIVAL_DELAY_CALC_1", "INT_ARRIVAL_DELAY_CALC_2"]
df_flights_considered[delay_cols] = df_flights_considered[delay_cols].astype(int)

In [ ]:
(df_flights_considered["INT_ARRIVAL_DELAY_CALC_1"] != df_flights_considered["ARRIVAL_DELAY"]).sum()

In [ ]:
df_flights_considered.loc[
    df_flights_considered["INT_ARRIVAL_DELAY_CALC_1"] != df_flights_considered["ARRIVAL_DELAY"],
    ["AIRLINE", "FLIGHT_NUMBER", "TAIL_NUMBER", "ORIGIN_AIRPORT", "DESTINATION_AIRPORT", "DT_ARRIVAL_CALC", "ARRIVAL_TIME", "SCHEDULED_ARRIVAL", "ARRIVAL_DELAY", "INT_ARRIVAL_DELAY_CALC_1"]
]

In [ ]:
(df_flights_considered["INT_ARRIVAL_DELAY_CALC_2"] != df_flights_considered["ARRIVAL_DELAY"]).sum()

In [ ]:
df_flights_considered.loc[
    df_flights_considered["INT_ARRIVAL_DELAY_CALC_2"] != df_flights_considered["ARRIVAL_DELAY"],
    ["AIRLINE", "FLIGHT_NUMBER", "TAIL_NUMBER", "ORIGIN_AIRPORT", "DESTINATION_AIRPORT", "DT_ARRIVAL_CALC", "ARRIVAL_TIME", "SCHEDULED_ARRIVAL", "ARRIVAL_DELAY", "INT_ARRIVAL_DELAY_CALC_2"]
]

## Delayed

In [ ]:
tol_min = 0
df_flights_considered["DELAYED"] = df_flights_considered["ARRIVAL_DELAY"] > tol_min

In [ ]:
df_flights_considered["DELAYED"].value_counts(normalize=True)

In [ ]:
delay_reason_cols = ["AIR_SYSTEM_DELAY", "SECURITY_DELAY", "AIRLINE_DELAY", "LATE_AIRCRAFT_DELAY", "WEATHER_DELAY"]
df_flights_considered.loc[df_flights_considered["DELAYED"], delay_reason_cols].isna().sum()

In [ ]:
df_flights_considered.loc[df_flights_considered["DELAYED"], ["ARRIVAL_DELAY"] + delay_reason_cols].sample(n=sample_size)

In [ ]:
df_flights_considered.loc[df_flights_considered["DELAYED"], delay_reason_cols].sum(axis=1, skipna=False).value_counts().sort_index()

In [ ]:
tol_min = 15
df_flights_considered["DELAYED"] = df_flights_considered["ARRIVAL_DELAY"] >= tol_min

In [ ]:
df_flights_considered["DELAYED"].value_counts(normalize=True)

In [ ]:
df_flights_considered.loc[df_flights_considered["DELAYED"], delay_reason_cols].isna().sum()

In [ ]:
df_flights_considered.loc[~df_flights_considered["DELAYED"], delay_reason_cols].isna().sum()

In [ ]:
df_flights_considered[~df_flights_considered["DELAYED"]].shape[0]

In [ ]:
(df_flights_considered.loc[df_flights_considered["DELAYED"], delay_reason_cols].sum(axis=1).astype(int) != df_flights_considered.loc[df_flights_considered["DELAYED"], "ARRIVAL_DELAY"]).sum()

In [ ]:
fig, (axc, axs) = plt.subplots(1, 2, figsize=(16, 5))

df_flights_considered.loc[df_flights_considered["DELAYED"], delay_reason_cols].gt(0).sum().plot(kind="bar", ax=axc)
df_flights_considered.loc[df_flights_considered["DELAYED"], delay_reason_cols].sum().plot(kind="bar", ax=axs)

In [ ]:
plot_opts = dict(
    kind="bar",
    stacked=True,
    color=["deepskyblue", "gainsboro"],
    figsize=(8, 5),
)

In [ ]:
cols = ["MONTH", "DELAYED"]

df_flights_considered[cols].value_counts().unstack().iloc[:, [1, 0]].plot(**plot_opts)

In [ ]:
cols = ["DAY", "DELAYED"]

df_flights_considered[cols].value_counts().sort_index().unstack().iloc[:, [1, 0]].plot(**plot_opts)

In [ ]:
cols = ["DAY_OF_WEEK_ABBR", "DELAYED"]

df_flights_considered[cols].value_counts().sort_index().unstack().iloc[:, [1, 0]].plot(**plot_opts)

In [ ]:
fig, (axb, axl) = plt.subplots(1, 2, figsize=(16, 5))
plot_opts_custom = {k: v for k, v in plot_opts.items() if k != "figsize"}

cols = ["HOUR", "DELAYED"]

df_flights_considered[cols].value_counts().sort_index().unstack().iloc[:, [1, 0]].plot(**plot_opts_custom, ax=axb)
(df_flights_considered.loc[df_flights_considered["DELAYED"], "HOUR"].value_counts() / df_flights_considered["HOUR"].value_counts()).sort_index().plot(kind="line", ax=axl)

In [ ]:
cols = ["ORIGIN_AIRPORT", "DELAYED"]

df_flights_considered[cols].value_counts().unstack(fill_value=0).assign(sum_temp=lambda df: df[False] + df[True]).sort_values(by="sum_temp", ascending=False).head(25).iloc[:, [1, 0]].plot(**plot_opts)

In [ ]:
_ = topk_barplot(df_flights_considered.loc[df_flights_considered["DELAYED"], "ORIGIN_AIRPORT"], normalize=True, k=25, group_others=False)
plt.show()

In [ ]:
fig, (axc, axd) = plt.subplots(1, 2, figsize=(16, 5))

_ = cumsum_plot(df_flights_considered["ORIGIN_AIRPORT"], annotate=True, threshold=0.8, ax=axc)
_ = cumsum_plot(df_flights_considered.loc[df_flights_considered["DELAYED"], "ORIGIN_AIRPORT"], annotate=True, threshold=0.8, ax=axd)

In [ ]:
cols = ["DESTINATION_AIRPORT", "DELAYED"]

df_flights_considered[cols].value_counts().unstack(fill_value=0).assign(sum_temp=lambda df: df[False] + df[True]).sort_values(by="sum_temp", ascending=False).head(25).iloc[:, [1, 0]].plot(**plot_opts)

In [ ]:
_ = topk_barplot(df_flights_considered.loc[df_flights_considered["DELAYED"], "DESTINATION_AIRPORT"], normalize=True, k=25, group_others=False)
plt.show()

In [ ]:
fig, (axc, axd) = plt.subplots(1, 2, figsize=(16, 5))

_ = cumsum_plot(df_flights_considered["DESTINATION_AIRPORT"], annotate=True, threshold=0.8, ax=axc)
_ = cumsum_plot(df_flights_considered.loc[df_flights_considered["DELAYED"], "DESTINATION_AIRPORT"], annotate=True, threshold=0.8, ax=axd)

In [ ]:
cols = ["ROUTE", "DELAYED"]

df_flights_considered[cols].value_counts().unstack(fill_value=0).assign(sum_temp=lambda df: df[False] + df[True]).sort_values(by="sum_temp", ascending=False).head(25).iloc[:, [1, 0]].plot(**plot_opts)

In [ ]:
_ = topk_barplot(df_flights_considered.loc[df_flights_considered["DELAYED"], "ROUTE"], normalize=True, k=25, group_others=False)
plt.show()

In [ ]:
fig, (axc, axd) = plt.subplots(1, 2, figsize=(16, 5))

_ = cumsum_plot(df_flights_considered["ROUTE"], annotate=True, threshold=0.8, ax=axc)
_ = cumsum_plot(df_flights_considered.loc[df_flights_considered["DELAYED"], "ROUTE"], annotate=True, threshold=0.8, ax=axd)

In [ ]:
fig, (axb, axl) = plt.subplots(1, 2, figsize=(16, 5))
plot_opts_custom = {k: v for k, v in plot_opts.items() if k != "figsize"}

cols = ["AIRLINE", "DELAYED"]

df_flights_considered[cols].value_counts().unstack(fill_value=0).sort_index().iloc[:, [1, 0]].plot(**plot_opts_custom, ax=axb)
airline_ratios = (df_flights_considered.loc[df_flights_considered["DELAYED"], "AIRLINE"].value_counts() / df_flights_considered["AIRLINE"].value_counts()).sort_index()
airline_ratios.plot(kind="line", xticks=range(len(airline_ratios)), use_index=True, rot=90, ax=axl)

In [ ]:
cols = ["FLIGHT_NUMBER", "DELAYED"]

df_flights_considered[cols].value_counts().unstack(fill_value=0).assign(sum_temp=lambda df: df[False] + df[True]).sort_values(by="sum_temp", ascending=False).head(25).iloc[:, [1, 0]].plot(**plot_opts)

In [ ]:
_ = topk_barplot(df_flights_considered.loc[df_flights_considered["DELAYED"], "FLIGHT_NUMBER"], normalize=True, k=25, group_others=False)
plt.show()

In [ ]:
fig, (axc, axd) = plt.subplots(1, 2, figsize=(16, 5))

_ = cumsum_plot(df_flights_considered["FLIGHT_NUMBER"], annotate=True, threshold=0.8, ax=axc)
_ = cumsum_plot(df_flights_considered.loc[df_flights_considered["DELAYED"], "FLIGHT_NUMBER"], annotate=True, threshold=0.8, ax=axd)

In [ ]:
cols = ["TAIL_NUMBER", "DELAYED"]

df_flights_considered[cols].value_counts().unstack(fill_value=0).assign(sum_temp=lambda df: df[False] + df[True]).sort_values(by="sum_temp", ascending=False).head(25).iloc[:, [1, 0]].plot(**plot_opts)

In [ ]:
_ = topk_barplot(df_flights_considered.loc[df_flights_considered["DELAYED"], "TAIL_NUMBER"], normalize=True, k=25, group_others=False)
plt.show()

In [ ]:
fig, (axc, axd) = plt.subplots(1, 2, figsize=(16, 5))

_ = cumsum_plot(df_flights_considered["TAIL_NUMBER"], annotate=True, threshold=0.8, ax=axc)
_ = cumsum_plot(df_flights_considered.loc[df_flights_considered["DELAYED"], "TAIL_NUMBER"], annotate=True, threshold=0.8, ax=axd)

In [ ]:
fig, (axb, axl) = plt.subplots(1, 2, figsize=(16, 5))
plot_opts_custom = {k: v for k, v in plot_opts.items() if k != "figsize"}

cols = ["FLIGHT_SEQUENCE", "DELAYED"]

df_flights_considered[cols].value_counts().sort_index().unstack().iloc[:, [1, 0]].plot(**plot_opts_custom, ax=axb)
(df_flights_considered.loc[df_flights_considered["DELAYED"], "FLIGHT_SEQUENCE"].value_counts() / df_flights_considered["FLIGHT_SEQUENCE"].value_counts()).sort_index().plot(kind="line", ax=axl)

In [ ]:
df_flights_considered["ARRIVAL_DELAY"].plot(kind="hist", bins=100, figsize=(8, 5))

In [ ]:
df_flights_considered["ARRIVAL_DELAY"].describe()

In [ ]:
cols = [
    x := "MONTH",
    y := "ARRIVAL_DELAY",
]

sns.violinplot(data=df_flights_considered[cols], x=x, y=y)

In [ ]:
cols = [
    x := "DAY",
    y := "ARRIVAL_DELAY",
]

sns.violinplot(data=df_flights_considered[cols], x=x, y=y)

In [ ]:
cols = [
    x := "DAY_OF_WEEK_ABBR",
    y := "ARRIVAL_DELAY",
]

sns.violinplot(data=df_flights_considered[cols], x=x, y=y)

In [ ]:
cols = [
    x := "HOUR",
    y := "ARRIVAL_DELAY",
]

sns.violinplot(data=df_flights_considered[cols], x=x, y=y)

In [ ]:
cols = [
    x := "AIRLINE",
    y := "ARRIVAL_DELAY",
]

sns.violinplot(data=df_flights_considered[cols], x=x, y=y)

In [ ]:
cols = [
    x := "FLIGHT_SEQUENCE",
    y := "ARRIVAL_DELAY",
]

sns.violinplot(data=df_flights_considered[cols], x=x, y=y)

In [ ]:
cols = [
    x := "DISTANCE",
    y := "ARRIVAL_DELAY",
]

df_flights_considered[cols].plot(kind="scatter", x=x, y=y, figsize=(8, 5))

In [ ]:
cols = [
    x := "SCHEDULED_TIME",
    y := "ARRIVAL_DELAY",
]

df_flights_considered[cols].plot(kind="scatter", x=x, y=y, figsize=(8, 5))